In [4]:
%reload_ext autoreload
%autoreload 2

In [5]:
import datetime
import json
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import wandb
from accelerate.commands.config.update import description
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

def set_seed(seed=42) -> None:
    """Set all seeds to make results reproducible (deterministic mode).
    When seed is a false-y value or not supplied, disables deterministic mode."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.append('/rhome/sawale/indus_traning/mlm-fine-tuning/mlm')

In [6]:
for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_properties(i).name)

# os.environ["MY_VARIABLE"] = "my_value"


NVIDIA RTX A5000
NVIDIA RTX A5000
NVIDIA RTX A5000
NVIDIA RTX A5000


## Load trained model from wandb

In [7]:
def load_model_from_wandb(model_path):
    run = wandb.init()
    artifact = run.use_artifact(model_path, type='model')
    artifact_dir = artifact.download()
    model = AutoModelForMaskedLM.from_pretrained(artifact_dir)
    return model, artifact_dir

def load_model_from_local(model_path):
    model = AutoModelForMaskedLM.from_pretrained(model_path)
    return model, model_path

def load_model(model_path, use_wandb=True):
    if use_wandb:
        return load_model_from_wandb(model_path)
    else:
        return load_model_from_local(model_path)


# load models

## load initial model nasa-impact/nasa-smd-ibm-v0.1
# model_0, model_0_path = load_model_from_local("nasa-impact/nasa-smd-ibm-v0.1")
tokenizer_indus = AutoTokenizer.from_pretrained("nasa-impact/nasa-smd-ibm-v0.1")

## load initial model fine tuned from nasa-smd-ibm-v0.1 with 100k dataset
# model_1, model_1_path = load_model_from_wandb("nasa-impact/mlm-fine-tuning/model-ytxjrbhy:v1")

## load model fine tuned from modernbert-base with 96k dataset using dynamic keyword masking
# model_3, model_3_path = load_model_from_wandb("nasa-impact/mlm-fine-tuning/model-yl3xhgco:v2")

## load model fine tuned on 500k indus extended model
model_4, model_4_path = load_model_from_wandb("nasa-impact/mlm-fine-tuning/model-33ci04l7:v1")

## load model fine tuned from modernbert-base with 500k dataset
model_5, model_5_path = load_model_from_wandb("nasa-impact/mlm-fine-tuning/model-zymz8ir5:v1")
tokenizer_modernbert = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")


wandb: Currently logged in as: sajil (nasa-impact) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


wandb: Downloading large artifact model-33ci04l7:v1, 477.21MB. 4 files... 
wandb:   4 of 4 files downloaded.  
Done. 0:0:10.2
wandb: Downloading large artifact model-zymz8ir5:v1, 570.92MB. 4 files... 
wandb:   4 of 4 files downloaded.  
Done. 0:0:1.0
You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


## Load Data and split identiacally

In [8]:
from preprocess_data import preprocess_dataset, preprocess_dataset_with_kw_masking, disk_cache
with open("../config_indus.json", "r") as file:
    config_indus = json.load(file)

with open("../config_modernbert.json", "r") as file:
    config_modernbert = json.load(file)


data_src = "local"
n_rows = 5000

config_indus["input"]["dataset"]["kw_masking_type"] = {"static": True, "dynamic": False}
lm_dataset_indus, tokenizer_indus, data_collator_indus = preprocess_dataset_with_kw_masking(
        config_indus.get("input"),
        data_src,
        n_rows,
    )

config_modernbert["input"]["dataset"]["kw_masking_type"] = {"static": True, "dynamic": False}
lm_dataset_mbs, tokenizer_mbs, data_collator_mbs = preprocess_dataset_with_kw_masking(
        config_modernbert.get("input"),
        data_src,
        n_rows,
    )


model_dict = {
    # "original_indus_model": {
    #     "model": model_0_path,
    #     "tokenizer": tokenizer_indus,
    #     "dataset": lm_dataset_indus,
    #     "config": config_indus
    # },
    # "finetuned_indus_100k_SDE": {
    #     "model": model_1_path,
    #     "tokenizer": tokenizer_indus,
    #     "dataset": lm_dataset_indus,
    #     "config": config_indus
    # },
    # "finetuned_modernBERT_100k_SDE": {
    #     "model": model_3_path,
    #     "tokenizer": tokenizer_modernbert,
    #     "dataset": lm_dataset_mbs,
    #     "config": config_modernbert
    # },
    "finetuned_indus_context_extended_500K_SDE": {
        "model": model_4_path,
        "tokenizer": tokenizer_indus,
        "dataset": lm_dataset_indus,
        "config": config_indus
    },
    
    "finetuned_modernbert_500k_sde": {
        "model": model_5_path,
        "tokenizer": tokenizer_modernbert,
        "dataset": lm_dataset_mbs,
        "config": config_modernbert
    }
}

Loading from cache...
Loading from cache...


In [9]:
lm_dataset_indus

DatasetDict({
    train: Dataset({
        features: ['text', 'yake', 'input_ids', 'attention_mask', 'offset_mapping', 'length', 'overflow_to_sample_mapping', 'probability_matrix', 'labels', 'input_ids_clone'],
        num_rows: 10870
    })
    validation: Dataset({
        features: ['text', 'yake', 'input_ids', 'attention_mask', 'offset_mapping', 'length', 'overflow_to_sample_mapping', 'probability_matrix', 'labels', 'input_ids_clone'],
        num_rows: 10568
    })
    test: Dataset({
        features: ['text', 'yake', 'input_ids', 'attention_mask', 'offset_mapping', 'length', 'overflow_to_sample_mapping', 'probability_matrix', 'labels', 'input_ids_clone'],
        num_rows: 10362
    })
})

In [10]:
def get_accuracy(inference_df):
    _inference_df = inference_df.copy()
    top_n = [c for c in _inference_df.columns if "top" in c]

    _inference_df['predictions'] = _inference_df.apply(lambda row: [row[col].get('token_str') for col in top_n], axis=1)

    for i, top_i in enumerate(top_n):
        _inference_df[f'{top_i}_correct'] = _inference_df.apply(lambda row: int(row['target'] in row['predictions'][:i+1]), axis=1)

    accuracy = {top_i: float(round(_inference_df[f'{top_i}_correct'].mean(), 4)) for top_i in top_n}

    return accuracy

## Make Predictions on train and test data

In [12]:
from utils import generate_inference

In [13]:
os.makedirs("tmp/inferences", exist_ok=True)

topk_accuracies = {}
for model_name, d in model_dict.items():
        topk_accuracies[model_name] = {}
        # for data_split in ["train", "test"]:
        for data_split in ["test"]:
                print(f"Generating inferences for {model_name} on {data_split}")
                inference_df = generate_inference(
                        d["dataset"][data_split],
                        d["tokenizer"],
                        d["model"],
                        top_k=d["config"].get("output").get("inference").get("top_k"),
                        n_predictions=None,
                        max_length=d["config"].get("input").get("dataset").get("chunk_size"),
                        use_keyword_based_masking=True
                )
                accuracy = get_accuracy(inference_df)
                topk_accuracies[model_name][data_split] = accuracy
                print(accuracy)
                inference_df.to_parquet(f"tmp/inferences/{model_name}_{data_split}_inference.parquet", index=False)
        print("-"*100)
        


Generating inferences for finetuned_indus_context_extended_500K_SDE on test


Device set to use cuda:0


{'top1': 0.7814, 'top2': 0.8319, 'top3': 0.8548}
----------------------------------------------------------------------------------------------------
Generating inferences for finetuned_modernbert_500k_sde on test


Device set to use cuda:0


{'top1': 0.6499, 'top2': 0.7169, 'top3': 0.7469}
----------------------------------------------------------------------------------------------------


In [15]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Convert dictionary to a DataFrame
data = []
for model_name, splits in topk_accuracies.items():
    for data_split, accuracies in splits.items():
        for top_k, acc in accuracies.items():
            data.append([model_name, data_split, top_k, acc])

df = pd.DataFrame(data, columns=["Model", "Dataset", "Top-K", "Accuracy"])

# Define results directory
results_dir = "results"

# Function to ensure directory exists
def ensure_directory(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Created directory: {directory}")

# Ensure results directory exists
ensure_directory(results_dir)

# Set color palette for models
model_palette = sns.color_palette("Set2", n_colors=len(df["Model"].unique()))

# Function to plot accuracy grouped by Top-K and save results
def plot_topk_accuracies(df, dataset):
    plt.figure(figsize=(10, 6))
    df_subset = df[df["Dataset"] == dataset]
    
    ax = sns.barplot(
        data=df_subset, x="Top-K", y="Accuracy", hue="Model", palette=model_palette
    )
    
    # Display accuracy values on top of each bar with 3 decimal places
    for p in ax.patches:
        height = p.get_height()
        if height > 0:  # Only label bars with positive height
            ax.annotate(
                f"{height:.3f}",  # Format to 3 decimal places
                (p.get_x() + p.get_width() / 2.0, height),
                ha="center",
                va="bottom",
                fontsize=10,
                color="black",
                fontweight="bold"
            )
    
    plt.title(f"Top-K Accuracies for {dataset.capitalize()} Set")
    plt.ylim(0, 1)
    plt.xlabel("Top-K Accuracy")
    plt.ylabel("Accuracy")
    plt.legend(title="Model")
    plt.tight_layout()
    
    # Save plot
    plot_path = f"{results_dir}/{dataset}_topk_accuracies.png"
    plt.savefig(plot_path, dpi=300)
    print(f"Plot saved to {plot_path}")
    
    plt.close()

    # Save data table
    table_path = f"{results_dir}/{dataset}_topk_accuracies.csv"
    df_subset.to_csv(table_path, index=False)
    print(f"Data table saved to {table_path}")

# Plot and save for Train Set
plot_topk_accuracies(df, "train")

# Plot and save for Test Set
plot_topk_accuracies(df, "test")


/tmp/ipykernel_684444/586238728.py:57: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title="Model")


Plot saved to results/train_topk_accuracies.png
Data table saved to results/train_topk_accuracies.csv
Plot saved to results/test_topk_accuracies.png
Data table saved to results/test_topk_accuracies.csv
